# LangChain — From Raw API Calls to Composable LLM Apps

**Duration:** ~2.5–3 hours
**Running case study:** *TechMart*, an online electronics store building an AI **support + catalog assistant**

---

You already know how to talk to an LLM directly — `client.chat.completions.create(...)`, the Responses API, prompt caching, multimodal inputs. For a **single call**, that raw SDK is honestly all you need.

So why does almost every production LLM app you read about reach for **LangChain**?

Because real apps are never a single call. They look like this:

> *retrieve the right documents → format a prompt → call a model → parse the reply into a typed object → maybe call a tool → remember the conversation → and do all of it with streaming, retries, and the option to swap GPT for Claude next week.*

LangChain is the glue for exactly that. In this notebook we build TechMart's assistant piece by piece, and at **every step we put the raw-SDK way next to the LangChain way** so you can see precisely what it buys you — and where it's overkill.

| # | Pattern | What LangChain gives you |
|---|---------|--------------------------|
| 1 | First contact | One uniform interface over every model |
| 2 | Prompts + parsers + LCEL | Compose steps with `\|` |
| 3 | batch / stream / parallel | Every execution mode, for free |
| 4 | Structured output | Typed Pydantic objects, no JSON wrangling |
| 5 | Model portability | Swap providers in one line; add fallbacks |
| 6 | RAG | Ground answers in your own documents |
| 7 | Memory | Multi-turn conversations |
| 8 | Tools & agents | Let the model take actions |
| 9 | Capstone | All of it, in one assistant |

## 0 · Setup

A note on versions: this notebook uses **LangChain v1.x** (`langchain`, `langchain-core`, `langchain-openai`, `langgraph`). The v1 line reorganized things — agents now run on **LangGraph**, and legacy chains like `LLMChain` / `RetrievalQA` are gone. If you hit an old tutorial using those, it predates 1.0.

In [1]:
# !pip install langchain langchain-openai langchain-community langgraph python-dotenv

import os, json, time, textwrap, warnings
warnings.filterwarnings("ignore")              # keep teaching output clean

from dotenv import load_dotenv
import truststore; truststore.inject_into_ssl()   # plays nice with corporate TLS / proxies

# ── compatibility shim (safe to delete on Colab / clean machines) ───────────────
# On some setups `langchain-text-splitters` eagerly imports sentence-transformers,
# which imports torchcodec, which needs a specific ffmpeg build. We do no audio/video
# here, so if the real torchcodec can't load, drop in a harmless stub.
import sys, types, importlib.machinery
try:
    import torchcodec  # noqa: F401
except Exception:
    def _stub(name, **attrs):
        m = types.ModuleType(name)
        m.__spec__ = importlib.machinery.ModuleSpec(name, loader=None)
        for k, v in attrs.items():
            setattr(m, k, v)
        sys.modules[name] = m
        return m
    _tc = _stub("torchcodec")
    _tc.decoders = _stub("torchcodec.decoders", AudioDecoder=object, VideoDecoder=object)
    for _s in ("encoders", "samplers", "transforms"):
        setattr(_tc, _s, _stub("torchcodec." + _s))

# ── API key ─────────────────────────────────────────────────────────────────────
load_dotenv("/Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/openai_key.env")
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY (e.g. in openai_key.env)"

MODEL = "gpt-5-nano"   # cheap + fast; everything here works on any chat model

def show(text, width=90):
    "Pretty-print a long string wrapped to the terminal width."
    print(textwrap.fill(str(text), width=width))

print("Setup OK - model:", MODEL)

Setup OK - model: gpt-5-nano


## 1 · First contact: the same call, two ways  (~15 min)

Start at the smallest unit: one prompt, one response. Here's TechMart turning a terse product spec into friendly copy — first with the **OpenAI SDK you already know**.

In [2]:
from openai import OpenAI
client = OpenAI()

blurb = "ANC over-ear BT5.3 headphones, 30h batt, foldable, multipoint."

resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You rewrite terse product specs into one friendly sentence."},
        {"role": "user",   "content": blurb},
    ],
)
print(resp.choices[0].message.content)

These Bluetooth 5.3 ANC over-ear headphones deliver up to 30 hours of battery life, feature a foldable design for easy portability, and support multipoint pairing.


Now the **LangChain** way. The model wrapper is `ChatOpenAI`. You hand `.invoke()` the same list of messages and get back a message object; the text lives in `.content`.

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODEL, temperature=0)

ai_msg = llm.invoke([
    {"role": "system", "content": "You rewrite terse product specs into one friendly sentence."},
    {"role": "user",   "content": blurb},
])
print(ai_msg.content)

Meet these ANC over-ear headphones with Bluetooth 5.3, up to 30 hours of battery life, a foldable design for easy portability, and multipoint pairing.


At this size the two are basically identical — a few characters shorter, nothing to write home about. **If your app is genuinely one call, stop here and use the raw SDK.**

What matters is what `llm` *is*. It's a **Runnable** — LangChain's single interface that every component implements:

- `.invoke(x)` — run once
- `.batch([x, y, z])` — run many, concurrently
- `.stream(x)` — stream the output as it's generated
- `.ainvoke` / `.abatch` / `.astream` — async versions of each

Prompts, parsers, retrievers, whole chains, even agents are **all Runnables** with these same methods. That uniformity is the whole game: learn the interface once and you compose pieces without learning a new API for each. And everything you know about tokens is still right there:

In [4]:
print(ai_msg.usage_metadata)

{'input_tokens': 40, 'output_tokens': 364, 'total_tokens': 404, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 320}}


## 2 · Prompt templates, parsers, and the `|` operator  (~20 min)

Two frictions appear the moment you have more than one call:

1. **Prompts get duplicated** — you paste the same instructions and f-string variables in, everywhere.
2. **Outputs need post-processing** — you rarely want the raw message object; you want a string, JSON, or a typed object.

LangChain's answer is three small pieces that snap together.

**(a) `ChatPromptTemplate`** — a reusable, parameterized prompt. `{curly}` slots get filled at call time.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are TechMart's copywriter. Rewrite the spec as {style} copy, one sentence."),
    ("human",  "{spec}"),
])

output = rewrite_prompt.invoke({"style": "playful", "spec": blurb})


# Fill it in to see exactly what the model will receive:
for m in rewrite_prompt.invoke({"style": "playful", "spec": blurb}).messages:
    print(f"{m.type:6s}: {m.content}")


system: You are TechMart's copywriter. Rewrite the spec as playful copy, one sentence.
human : ANC over-ear BT5.3 headphones, 30h batt, foldable, multipoint.


**(b) Output parsers** turn the reply into something usable. The simplest, `StrOutputParser`, just pulls out `.content`.

**(c) LCEL — the pipe.** This is the line that makes people *like* LangChain. Connect Runnables with `|`, exactly like a Unix pipe; the output of each stage flows into the next:

```
prompt | model | parser
```

In [ ]:
#input dictionary
#     ↓
#rewrite_prompt
#     ↓
#formatted chat messages
#     ↓
#llm
#     ↓
#AIMessage
#     ↓
#StrOutputParser
#     ↓
#plain string

In [15]:
from langchain_core.output_parsers import StrOutputParser

copywriter = rewrite_prompt | llm | StrOutputParser()

print(copywriter.invoke({"style": "playful", "spec": blurb}))

Block out the world with ANC over-ear headphones—Bluetooth 5.3, a 30-hour battery, foldable for travel, and multipoint pairing to switch devices in a snap.


Read it left to right: a dict goes in → `rewrite_prompt` formats the messages → `llm` calls the model → `StrOutputParser` returns a clean string. (This is the modern replacement for the old `LLMChain`.)

Crucially, `copywriter` is **itself a Runnable**. So it has `.invoke`, `.batch`, and `.stream` too — which is the entire point of the next section.

## 3 · One chain, every execution mode — for free  (~15 min)

Here's the first place LangChain clearly beats hand-rolling. You wrote `copywriter` once. Without touching it, you get:

**Batch** — TechMart has a catalog, not one product. `.batch()` runs the inputs concurrently (the client-side cousin of the throughput ideas from the batching lesson).

In [16]:
specs = [
    {"style": "playful",    "spec": "ANC over-ear BT5.3 headphones, 30h batt, foldable."},
    {"style": "premium",    "spec": "Ergo office chair, 4D arms, mesh back, 135-degree recline."},
    {"style": "minimalist", "spec": "1L vacuum steel bottle, 24h cold, leakproof."},
]
for line in copywriter.batch(specs):
    print("-", line)

- Block out the world in style with ANC over-ear headphones—Bluetooth 5.3, a 30-hour battery, and foldable for tunes on the go.
- Meet the Ergo office chair—a premium ergonomic solution with 4D adjustable arms, a breathable mesh back, and a 135-degree recline for personalized comfort and support.
- 1L vacuum steel bottle keeps drinks cold for 24 hours and is leakproof.


**Stream** — for a chat UI you want tokens to appear as they're produced, not after a 3-second wait. Same chain, `.stream()`:

In [17]:
for chunk in copywriter.stream({"style": "energetic", "spec": "27-inch 4K monitor, 144Hz, USB-C 90W."}):
    print(chunk, end="", flush=True)

Unleash ultra-crisp 4K detail and blistering 144Hz on a 27-inch monitor with USB-C 90W PD for single-cable power, data, and charging.

**Parallel** — run several chains on the same input at once with `RunnableParallel` (a plain dict is shorthand). TechMart wants a tagline **and** keywords for each product in one shot:

In [18]:
from langchain_core.runnables import RunnableParallel

tagline  = ChatPromptTemplate.from_template("One catchy 6-word tagline for: {spec}") | llm | StrOutputParser()
keywords = ChatPromptTemplate.from_template("Five comma-separated SEO keywords for: {spec}") | llm | StrOutputParser()

enrich = RunnableParallel(
    tagline_output=tagline,
    keywords_output=keywords
)

out = enrich.invoke({
    "spec": "ANC over-ear BT5.3 headphones, 30h batt, foldable."
})

print("TAGLINE :", out["tagline_output"])
print("KEYWORDS:", out["keywords_output"])

TAGLINE : ANC over-ear BT5.3, 30h battery, foldable.
KEYWORDS: ANC over-ear headphones, BT5.3 headphones, 30-hour battery life headphones, foldable wireless headphones, noise-cancelling headphones


The two model calls inside `enrich` run **concurrently**, not one after the other. Async (`.ainvoke`) works the same way — write the chain once, run it however you need.

## 4 · Structured output: stop parsing JSON by hand  (~15 min)

TechMart's support inbox needs every message tagged: *what category, how urgent, one-line summary.*

With the raw SDK you'd hand-write a JSON schema (or beg the model in the prompt), then `json.loads` the reply and hope it's valid. LangChain collapses that to: **define a Pydantic model, call `.with_structured_output()`, and get a validated, typed object back** — no parsing, no try/except.

In [10]:
from pydantic import BaseModel, Field
from typing import Literal

class TicketTag(BaseModel):
    "Structured triage for one support message."
    category: Literal["billing", "shipping", "technical", "returns", "other"]
    urgency: int = Field(description="1 (can wait) to 5 (on fire)", ge=1, le=5)
    summary: str = Field(description="one-line summary for the support queue")

triage = llm.with_structured_output(TicketTag)

tag = triage.invoke("My order says delivered but the box is empty and I leave on a trip tomorrow!")
print(type(tag).__name__)
print(tag)

TicketTag
category='shipping' urgency=5 summary='Order marked delivered but box is empty before trip.'


`tag` is a real `TicketTag` instance: `tag.urgency` is an `int`, `tag.category` is guaranteed to be one of the allowed values. Hand it straight to your database. And because `triage` is a Runnable, you can `.batch()` the whole inbox:

In [11]:
inbox = [
    "Where is my refund? It has been three weeks.",
    "The laptop will not turn on out of the box.",
    "Do you ship to Canada?",
    "URGENT: charged twice for order A1099!!",
]
for t in triage.batch(inbox):
    print(f"[{t.category:9s} u{t.urgency}]  {t.summary}")

[billing   u4]  Customer inquiring about delayed refund after three weeks.
[technical u5]  Laptop won't power on after unboxing.
[shipping  u1]  Inquiry about shipping availability to Canada.
[billing   u5]  Customer charged twice for order A1099


Under the hood this uses the model's native tool/function-calling to **guarantee the shape** — the same mechanism you met in function-calling, except you never touch the schema plumbing.

## 5 · Swap models in one line — the feature teams stay for  (~10 min)

This is the pitch that wins over engineering leads. Your `copywriter` and `triage` chains never say "OpenAI" anywhere except where you built `llm`. Swap that one object and **every chain downstream uses the new model** — no other edits.

`init_chat_model` is the universal constructor: give it a model name and a provider.

In [12]:
from langchain.chat_models import init_chat_model

llm_mini = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0)
llm_4o   = init_chat_model("gpt-4o",      model_provider="openai", temperature=0)

q = "In one sentence: why might a customer return wireless earbuds?"
print("mini:", llm_mini.invoke(q).content)
print("4o  :", llm_4o.invoke(q).content)

mini: A customer might return wireless earbuds due to poor sound quality, connectivity issues, or discomfort during use.


4o  : A customer might return wireless earbuds due to poor sound quality, connectivity issues, discomfort, or a defect in the product.


Same method, different brain. Moving a chain to Anthropic or Google changes **only** the constructor (keys permitting) — the prompts, parsers, RAG, and agents downstream don't change at all:

```python
# claude = init_chat_model("claude-sonnet-4-5", model_provider="anthropic")
# gemini = init_chat_model("gemini-2.0-flash",  model_provider="google_genai")
```

**Fallbacks** are the other half of resilience. `.with_fallbacks()` returns a Runnable that tries a backup if the primary errors — outage, rate limit, bad key. Here the primary uses a deliberately broken key:

In [13]:
flaky  = init_chat_model("gpt-4o-mini", model_provider="openai",
                         api_key="sk-intentionally-broken", max_retries=0)
backup = init_chat_model("gpt-4o-mini", model_provider="openai")

resilient = flaky.with_fallbacks([backup])
print(resilient.invoke("Reply with the single word: online").content)

Online


The primary fails on the bad key, LangChain silently falls back to `backup`, and the user never sees an error. Hand-wiring that around every provider call is precisely the boilerplate LangChain absorbs.

## 6 · RAG: answer from TechMart's own documents  (~25 min)

Ask a plain model "what's TechMart's return window?" and it *guesses* — it has never seen TechMart's policies. The fix is **Retrieval-Augmented Generation**: store your documents, fetch the few relevant chunks at question time, and drop them into the prompt as grounding.

The pipeline is five moves, and LangChain has a primitive for each:

| Step | Primitive |
|------|-----------|
| break docs into chunks | `RecursiveCharacterTextSplitter` |
| turn chunks into vectors | `OpenAIEmbeddings` |
| store + search vectors | `InMemoryVectorStore` |
| fetch top-k for a query | `.as_retriever()` |
| stuff context + answer | an LCEL chain |

First, TechMart's (tiny) knowledge base:

In [14]:
from langchain_core.documents import Document

policy_text = """
TechMart Return Policy
Most items can be returned within 30 days of delivery for a full refund. Opened software and gift
cards are non-refundable. Return shipping is free for defective items; otherwise a $6 label fee applies.

TechMart Shipping
Standard shipping is free on orders over $50 and takes 3-5 business days. Express shipping is $15 and
arrives in 1-2 business days. We currently ship within the US and Canada only.

TechMart Warranty
All electronics include a 1-year manufacturer warranty covering defects in materials and workmanship.
Accidental damage is not covered. Extended 3-year protection plans cost 12% of the item price at checkout.

TechMart Membership
TechMart Plus is $49/year and includes free express shipping, an extra 60 days of return window,
and early access to sales.
"""
docs = [Document(page_content=policy_text)]

**Split.** Long docs don't fit a prompt well (and dilute retrieval), so chunk them. `RecursiveCharacterTextSplitter` prefers to break on paragraph and line boundaries before resorting to mid-sentence cuts.

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=40)
chunks = splitter.split_documents(docs)
print(f"{len(chunks)} chunks. First chunk:\n")
show(chunks[0].page_content)

4 chunks. First chunk:

TechMart Return Policy Most items can be returned within 30 days of delivery for a full
refund. Opened software and gift cards are non-refundable. Return shipping is free for
defective items; otherwise a $6 label fee applies.


**Embed + store.** `OpenAIEmbeddings` turns each chunk into a vector; `InMemoryVectorStore` holds them and does similarity search. (`check_embedding_ctx_length=False` skips a local tokenizer download — fine for short docs.) A retriever is a Runnable too — give it a question, get back the most relevant `Document`s.

In [16]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embeddings   = OpenAIEmbeddings(model="text-embedding-3-small", check_embedding_ctx_length=False)
vectorstore  = InMemoryVectorStore.from_documents(chunks, embeddings)
retriever    = vectorstore.as_retriever(search_kwargs={"k": 2})

for d in retriever.invoke("how long do I have to return something?"):
    print("-", d.page_content[:80].strip(), "...")

- TechMart Return Policy
Most items can be returned within 30 days of delivery for ...
- TechMart Warranty
All electronics include a 1-year manufacturer warranty coverin ...


**Assemble the RAG chain.** The only new trick is feeding two things into the prompt — the retrieved `context` and the user's `question`. We do that with a dict at the front of the chain; `RunnablePassthrough` forwards the original question through untouched.

In [17]:
from langchain_core.runnables import RunnablePassthrough

rag_prompt = ChatPromptTemplate.from_template(
    "You are TechMart support. Answer the question using ONLY the context.\n"
    "If the answer is not in the context, say you do not know.\n\n"
    "Context:\n{context}\n\nQuestion: {question}"
)

def format_docs(ds):
    return "\n\n".join(d.page_content for d in ds)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

for question in ["What's the return window?",
                 "Do you ship to Canada?",
                 "Is accidental damage covered by the warranty?"]:
    print("Q:", question)
    print("A:", rag_chain.invoke(question), "\n")

Q: What's the return window?


A: The return window is 30 days of delivery for most items. If you have a TechMart Plus membership, it is extended to an extra 60 days. 

Q: Do you ship to Canada?


A: Yes, we currently ship to Canada. 

Q: Is accidental damage covered by the warranty?


A: No, accidental damage is not covered by the warranty. 



Grounded answers, with the source chunks available for citation. Compare that to doing it by hand: compute embeddings, store them, write cosine-similarity search, sort, slice top-k, assemble the prompt. Here it's a dozen lines — and swapping `InMemoryVectorStore` for **FAISS, Chroma, or Pinecone** is a one-line change because they all share the same interface.

## 7 · Memory: holding a conversation  (~15 min)

LLMs are **stateless** — each call forgets the last. "Does it come in black?" is meaningless without the previous turn. The raw-SDK fix is to keep appending to a `messages` list yourself and resend the whole thing every time.

LangChain prompts make room for that history with `MessagesPlaceholder`:

In [18]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are TechMart support. Be warm and concise."),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])
chat_chain = chat_prompt | llm | StrOutputParser()

history = [
    HumanMessage("I'm looking at the wireless earbuds."),
    AIMessage("Great pick! They come in three colors and have 8-hour battery life."),
]
print(chat_chain.invoke({"history": history, "input": "Do they come in black?"}))

Yes, the wireless earbuds do come in black! Would you like more information on their features?


That works, but **you** are babysitting the `history` list. For a real app — many users, each with their own thread, persisted across requests — LangChain's modern answer is a **LangGraph agent with a checkpointer**. Give each conversation a `thread_id` and the framework stores and replays history for you.

In [19]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

memory_bot = create_agent(
    llm,
    tools=[],
    system_prompt="You are TechMart support. Be warm and concise.",
    checkpointer=InMemorySaver(),
)

alice = {"configurable": {"thread_id": "alice"}}

def say(text, cfg):
    out = memory_bot.invoke({"messages": [{"role": "user", "content": text}]}, cfg)
    print(out["messages"][-1].content)

say("Hi, I'm Alice and I just bought the 4K monitor.", alice)
say("Remind me - what did I buy?", alice)   # remembers; we passed no history

Hi Alice! Congratulations on your new 4K monitor! If you have any questions about setup or features, feel free to ask. I'm here to help!


You purchased a 4K monitor. If you need details about its specifications or features, just let me know!


We never re-sent the first message — the checkpointer did. Start a different `thread_id` and it's a clean slate: that's **per-user memory with zero bookkeeping** on your side. (`InMemorySaver` is for demos; swap in a SQLite or Postgres saver and the same code survives restarts.)

## 8 · Tools and agents: let the assistant *do* things  (~25 min)

RAG answers from documents. But "where's my order A1099?" needs a **live lookup**, not a document. For that the model has to call your code — a **tool** — read the result, and decide what to do next. That decide → act → observe loop is an **agent**.

In LangChain v1, `@tool` turns a function into a tool (its docstring tells the model *when* to use it), and `create_agent` wires up the loop.

In [20]:
from langchain_core.tools import tool

# Pretend these hit TechMart's databases:
_ORDERS = {"A1099": "shipped - arriving Thursday via FedEx",
           "A1023": "processing - ships tomorrow"}
_STOCK  = {"wireless earbuds": 0, "4k monitor": 12, "office chair": 5}

@tool
def get_order_status(order_id: str) -> str:
    "Look up the live shipping status of a TechMart order by its id, e.g. 'A1099'."
    return _ORDERS.get(order_id.upper().strip(), "no order with that id")

@tool
def check_stock(product: str) -> str:
    "Return how many units of a product are currently in stock."
    n = _STOCK.get(product.lower().strip())
    if n is None:
        return "unknown product"
    return f"{n} in stock" if n else "out of stock"

Hand the tools to an agent and ask something that needs one:

In [21]:
ops_agent = create_agent(llm, tools=[get_order_status, check_stock])

result = ops_agent.invoke({"messages": [
    {"role": "user", "content": "Is order A1099 on its way yet?"}
]})
print(result["messages"][-1].content)

Yes, order A1099 has been shipped and is expected to arrive on Thursday via FedEx.


To see *why* it answered that, walk the message trace: the model emits a **tool call**, the tool's result comes back, then the model writes the final answer.

In [22]:
def trace(result):
    for m in result["messages"]:
        if getattr(m, "tool_calls", None):
            for tc in m.tool_calls:
                print(f"  [ai -> tool] {tc['name']}({tc['args']})")
        elif m.type == "tool":
            print(f"  [tool -> ai] {m.content}")
        elif m.type == "human":
            print(f"[user] {m.content}")
        elif m.content:
            print(f"[ai] {m.content}")

trace(result)

[user] Is order A1099 on its way yet?
  [ai -> tool] get_order_status({'order_id': 'A1099'})
  [tool -> ai] shipped - arriving Thursday via FedEx
[ai] Yes, order A1099 has been shipped and is expected to arrive on Thursday via FedEx.


The real power shows when a question needs **several** tools and the agent sequences them itself — no `if/else` from you:

In [23]:
multi = ops_agent.invoke({"messages": [{"role": "user", "content":
    "Are the wireless earbuds in stock, and where is order A1023?"}]})
trace(multi)

[user] Are the wireless earbuds in stock, and where is order A1023?
  [ai -> tool] check_stock({'product': 'wireless earbuds'})
  [ai -> tool] get_order_status({'order_id': 'A1023'})
  [tool -> ai] out of stock
  [tool -> ai] processing - ships tomorrow
[ai] The wireless earbuds are currently out of stock. As for order A1023, it is processing and is scheduled to ship tomorrow.


You wrote two plain Python functions. The agent decided **which** to call, in **what order**, with **what arguments**, and how to fold the results into one answer. That orchestration — tedious and bug-prone by hand — is what `create_agent` hands you.

## 9 · Capstone: the whole TechMart assistant  (~15 min)

Now combine everything: an agent that can **search policies (RAG)**, **look up live data (tools)**, and **remember the conversation (checkpointer)** — one assistant, one `create_agent` call.

First, expose the RAG retriever as a tool so the agent can *choose* to consult the policy docs:

In [24]:
@tool
def search_policies(query: str) -> str:
    "Search TechMart policy docs (returns, shipping, warranty, membership) for an answer."
    return format_docs(retriever.invoke(query))

support_agent = create_agent(
    llm,
    tools=[get_order_status, check_stock, search_policies],
    system_prompt=(
        "You are TechMart's support assistant. Use tools to look up live order and stock "
        "info and to check policies. Be concise and friendly."
    ),
    checkpointer=InMemorySaver(),
)

In [25]:
bob = {"configurable": {"thread_id": "bob"}}

def ask(text):
    out = support_agent.invoke({"messages": [{"role": "user", "content": text}]}, bob)
    print("USER:", text)
    print("BOT :", out["messages"][-1].content, "\n")

ask("Hi! Are the wireless earbuds in stock?")
ask("Shame. If I had ordered them, what's the return window if they don't fit my ears?")
ask("Also, where is my order A1099?")

USER: Hi! Are the wireless earbuds in stock?
BOT : I'm sorry, but the wireless earbuds are currently out of stock. If you have any other questions or need assistance with something else, feel free to ask! 



USER: Shame. If I had ordered them, what's the return window if they don't fit my ears?
BOT : If you ordered the wireless earbuds, you can return them within 30 days of delivery for a full refund. If you're a TechMart Plus member, you would have an extended return window of 60 days. Let me know if you need more information! 



USER: Also, where is my order A1099?
BOT : Your order A1099 has been shipped and is scheduled to arrive on Thursday via FedEx. If you have any more questions or need further assistance, just let me know! 



One conversation that crossed **inventory**, **policy retrieval**, and **order lookup** — while remembering context between turns (notice "them" resolved to the earbuds). That's a genuinely useful support bot in ~15 lines of glue.

## When *not* to reach for LangChain

Keep your judgment calibrated — LangChain is a dependency and an abstraction, not a free lunch:

- **One prompt in, one string out, no composition?** Use the raw SDK. LangChain earns its keep when you have *steps*.
- **Debugging a provider quirk and need to see the exact bytes on the wire?** Drop to the SDK for that call.
- **Want total control of an agent loop?** Use LangGraph directly — `create_agent` is the sensible default, not the only option.

Where it clearly pays off — and what we used today:

| You want to… | LangChain primitive | Why it's simpler |
|--------------|---------------------|------------------|
| call any model the same way | `ChatOpenAI` / `init_chat_model` | one interface, swappable provider |
| reuse parameterized prompts | `ChatPromptTemplate` | no copy-pasted f-strings |
| compose steps | LCEL `\|` | readable left-to-right pipelines |
| run many / stream / async | `.batch` `.stream` `.ainvoke` | written once, free everywhere |
| typed results | `.with_structured_output` | Pydantic objects, no JSON parsing |
| survive outages | `.with_fallbacks` | automatic backup model |
| answer from your docs | splitter + embeddings + vector store | RAG in a dozen lines |
| hold a conversation | agent + checkpointer | per-user memory, no bookkeeping |
| take actions | `@tool` + `create_agent` | the model orchestrates your functions |

The thread running through all of it: **everything is a Runnable**, so everything composes. Learn the interface once and prompts, models, retrievers, tools, and agents all snap together with the same `|`.

### Try it yourself

- Add a `track_refund(order_id)` tool and ask the capstone agent a refund question.
- Swap `InMemoryVectorStore` for `FAISS` (one line) and confirm the RAG answers match.
- Give `TicketTag` a new field (e.g. `sentiment`) and re-run the inbox triage.
- Point `init_chat_model` at `gpt-4o` for the agent and see whether the tool-use trace changes.